In [ ]:
%cd ..

In [2]:
import os
import sys
import json
import time
import requests
import pandas as pd
from datetime import datetime
from requests.auth import HTTPBasicAuth


In [7]:
import re
import unicodedata
from collections import defaultdict

def normalize_vietnamese(text: str) -> str:
    """
    Bỏ dấu tiếng Việt
    """
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text


def to_sql_column_name(text: str) -> str:
    """
    Chuyển text → sql column name
    """
    if not text or not text.strip():
        return "col"

    text = text.strip().lower()
    text = normalize_vietnamese(text)

    # thay ký tự đặc biệt bằng _
    text = re.sub(r"[^a-z0-9]", "_", text)

    # gộp nhiều _ thành 1
    text = re.sub(r"_+", "_", text)

    # bỏ _ ở đầu/cuối
    text = text.strip("_")

    # không được bắt đầu bằng số
    if re.match(r"^\d", text):
        text = "_" + text

    return text or "col"


def generate_sql_columns(original_names):
    """
    Input:  list[str]  (tên cột gốc)
    Output:
        - list[str] sql column names
        - dict mapping original -> sql
    """
    used = defaultdict(int)
    mapping = {}
    sql_columns = []

    for name in original_names:
        base = to_sql_column_name(name)

        if used[base] == 0:
            sql_name = base
        else:
            sql_name = f"{base}_{used[base]}"

        used[base] += 1

        mapping[name] = sql_name
        sql_columns.append(sql_name)

    return sql_columns, mapping

def normalize_df_columns(df):
    """
    Input : pandas.DataFrame
    Output:
        - DataFrame đã rename
        - dict mapping: {original_name: sql_name}
    """
    used = defaultdict(int)
    mapping = {}

    new_columns = []

    for col in df.columns:
        base = to_sql_column_name(col)

        if used[base] == 0:
            new_col = base
        else:
            new_col = f"{base}_{used[base]}"

        used[base] += 1

        mapping[col] = new_col
        new_columns.append(new_col)

    df = df.copy()
    df.columns = new_columns

    return df, mapping

In [4]:
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
import os
import sys
import json
import time
import requests
import pandas as pd
from datetime import datetime


DATETIME_FORMATS = [
    "%Y-%m-%d",
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%d %H:%M:%S.%f",
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%dT%H:%M:%S.%f",
    "%Y-%m-%dT%H:%M:%SZ",
    "%Y-%m-%dT%H:%M:%S.%fZ",
    "%Y-%m-%dT%H:%M:%S%z",
    "%Y-%m-%dT%H:%M:%S.%f%z"
]
TIME_FIELD_NAMES = [ "created",
    "created_at",
    "updated",
    "updated_at"]
import pytz

def fix_empty_object_in_json(data, dummy_field="_dummy_"):
    """
    Fix empty object {} in JSON recursively
    """
    # dict
    if isinstance(data, dict):
        # object rỗng
        if len(data) == 0:
            return {dummy_field: 1}

        fixed = {}
        for k, v in data.items():
            fixed[k] = fix_empty_object_in_json(v, dummy_field)
        return fixed

    # list
    if isinstance(data, list):
        return [
            fix_empty_object_in_json(item, dummy_field)
            for item in data
        ]

    # primitive
    return data

def to_epoch_millis(value):
    if value is None:
        return None

    if isinstance(value, (int, float)):
        # assume already epoch
        return int(value)

    if not isinstance(value, str):
        return None

    for fmt in DATETIME_FORMATS:
        try:
            dt = datetime.strptime(value, fmt)

            # normalize timezone
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=pytz.UTC)

            return int(dt.timestamp() * 1000)
        except Exception:
            pass

    return None

def is_datetime_string(value):
    if not isinstance(value, str):
        return False

    for fmt in DATETIME_FORMATS:
        try:
            datetime.strptime(value, fmt)
            return True
        except Exception:
            pass

    return False


def convert_json_list_time_to_long(records):
    """
    records: List[dict]
    return: List[dict]
    """
    output = []

    for r in records:
        new_r = {}

        for k, v in r.items():
            key_lower = k.lower()

            if (
                key_lower in TIME_FIELD_NAMES
                or is_datetime_string(v)
            ):
                new_r[k] = to_epoch_millis(v)
            else:
                new_r[k] = v

        output.append(new_r)

    return output


def parse_datetime_to_epoch(value):
    if value is None:
        return None

    if isinstance(value, (int, float)):
        return int(value)

    if not isinstance(value, str):
        return None

    for fmt in DATETIME_FORMATS:
        try:
            dt = datetime.strptime(value, fmt)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=pytz.UTC)
            return int(dt.timestamp() * 1000)
        except Exception:
            pass

    return None


def convert_value_by_arrow_type(value, arrow_type: pa.DataType):
    if value is None:
        return None

    # primitive
    if pa.types.is_int64(arrow_type):
        return int(value) if value is not None else None

    if pa.types.is_float64(arrow_type):
        return float(value) if value is not None else None

    if pa.types.is_boolean(arrow_type):
        return bool(value)

    if pa.types.is_string(arrow_type):
        return str(value)

    if pa.types.is_timestamp(arrow_type):
        return parse_datetime_to_epoch(value)

    # complex
    if pa.types.is_struct(arrow_type):
        if not isinstance(value, dict):
            return None
        return {
            f.name: convert_value_by_arrow_type(
                value.get(f.name),
                f.type
            )
            for f in arrow_type
        }

    if pa.types.is_list(arrow_type):
        if not isinstance(value, list):
            return None
        return [
            convert_value_by_arrow_type(v, arrow_type.value_type)
            for v in value
        ]

    return value

def convert_record_by_arrow_schema(record: dict, schema: pa.Schema):
    out = {}
    for field in schema:
        out[field.name] = convert_value_by_arrow_type(
            record.get(field.name),
            field.type
        )
    return out

def convert_json_list_by_arrow_schema(records, schema: pa.Schema):
    return [
        convert_record_by_arrow_schema(r, schema)
        for r in records
    ]
    
    
def infer_arrow_type(values):
    values = [v for v in values if v is not None]
    if not values:
        return pa.string()

    if all(isinstance(v, bool) for v in values):
        return pa.bool_()

    if all(isinstance(v, int) for v in values):
        return pa.int64()

    if all(isinstance(v, (int, float)) for v in values):
        return pa.float64()

    if all(isinstance(v, str) for v in values):
        # detect timestamp
        return pa.string()
        try:
            for v in values[:5]:
                parser.isoparse(v)
            return pa.timestamp("ms")
        except Exception:
            return pa.string()

    if all(isinstance(v, dict) for v in values):
        return infer_struct(values)

    if all(isinstance(v, list) for v in values):
        elem_type = infer_arrow_type(
            [e for v in values for e in v if e is not None]
        )
        return pa.list_(elem_type)

    return pa.string()


def infer_struct(dicts):
    fields = {}
    for d in dicts:
        for k, v in d.items():
            fields.setdefault(k, []).append(v)

    return pa.struct([
        pa.field(k, infer_arrow_type(v))
        for k, v in fields.items()
    ])
    
    
def infer_schema_from_json(records):
    cols = {}
    for r in records:
        for k, v in r.items():
            cols.setdefault(k, []).append(v)

    return pa.schema([
        pa.field(k, infer_arrow_type(v))
        for k, v in cols.items()
    ])
    
def parse_datetime_to_epoch_ms(value):
    if not isinstance(value, str):
        return None

    for fmt in DATETIME_FORMATS:
        try:
            dt = datetime.strptime(value, fmt)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=pytz.UTC)
            return int(dt.timestamp() * 1000)
        except Exception:
            pass

    return None

def convert_json_add_ts_columns(records, suffix="_ts"):
    """
    records: List[dict]
    return: List[dict]
    """
    out = []

    for r in records:
        r = fix_empty_object_in_json(r)
        new_r = dict(r)

        for k, v in r.items():
            epoch = parse_datetime_to_epoch_ms(v)
            if epoch is not None:
                new_r[k + suffix] = epoch

        out.append(new_r)

    return out

    
def arrow_to_hive_type(t: pa.DataType) -> str:
    if pa.types.is_int64(t):
        return "BIGINT"
    if pa.types.is_float64(t):
        return "DOUBLE"
    if pa.types.is_boolean(t):
        return "BOOLEAN"
    if pa.types.is_timestamp(t):
        return "TIMESTAMP"
    if pa.types.is_string(t):
        return "STRING"
    if pa.types.is_struct(t):
        fields = [
            f"{f.name}:{arrow_to_hive_type(f.type)}"
            for f in t
        ]
        return f"STRUCT<{', '.join(fields)}>"
    if pa.types.is_list(t):
        return f"ARRAY<{arrow_to_hive_type(t.value_type)}>"
    return "STRING"


def gen_spark_create_table(schema, db, table, location):
    cols = []
    for f in schema:
        cols.append(
            "  {name} {dtype}".format(
                name=f.name,
                dtype=arrow_to_hive_type(f.type)
            )
        )

    sql = """
        CREATE TABLE IF NOT EXISTS {db}.{table} (
        {cols}
        )
        USING PARQUET
        LOCATION '{location}'
    """.format(
            db=db,
            table=table,
            cols=",\n".join(cols),
            location=location
        )

    return sql.strip()

In [5]:
AMBARI_SESSION = "SUPPORTSESSIONID=mjfqv09ao3s71j5xkdfyfmvfw; JSESSIONID=e2cb15e5-3942-4ef1-9df5-21aacd284ac9; AMBARISESSIONID=node02hvb5wrg4j9w1eyxet39i53p30.node0"

REQUEST_TIMEOUT = 10

# =========================
# HTTPS HDFS (Ambari Files View)
# =========================
AMBARI_BASE = "https://datalake.viettelcyber.com/gateway/ui/ambari"
FILES_API = "{}/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files".format(
    AMBARI_BASE
)

# AMBARI_SESSION = os.getenv("AMBARISESSIONID")
# if not AMBARI_SESSION:
    # raise Exception("Missing AMBARISESSIONID env var")

HDFS_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "X-Requested-By": "ambari",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type": "application/json",
    "Cookie": AMBARI_SESSION
}

HDFS_UPLOAD_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "X-Requested-By": "ambari",
    "X-Requested-With": "XMLHttpRequest",
    "Cookie": AMBARI_SESSION
}

HDFS_MKDIR_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "X-Requested-By": "ambari",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type":"application/json",
    "Cookie": AMBARI_SESSION
}


if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")
    
# =========================
# HDFS HTTPS helpers
# =========================
def read_hdfs_https(path):
    url = "{}/download/browse?download=true&?path={}".format(FILES_API, path)
    r = requests.get(url, headers=HDFS_HEADERS, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.text

def upload_hdfs_https(hdfs_path, local_file):
    print("Upload ", hdfs_path, local_file)
    mkdirs_hdfs_https(hdfs_path)
    
    url = "{}/upload".format(FILES_API)
    files = {
        "file": (os.path.basename(local_file), open(local_file, "rb")),
        "path": hdfs_path
    }
    r = requests.put(url, headers=HDFS_UPLOAD_HEADERS, files=files, data=None, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    
def mkdirs_hdfs_https(hdfs_path):
    url = "{}/fileops/mkdir".format(FILES_API)
    data = {
        "path": hdfs_path
    }
    r = requests.put(url, headers=HDFS_MKDIR_HEADERS, files=None, json=data, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    
    
def remove_hdfs_https(hdfs_path):
    url = "{}/fileops/remove".format(FILES_API)
    data = {"paths":[{"path": hdfs_path,"recursive":True}]}
    r = requests.post(url, headers=HDFS_MKDIR_HEADERS, files=None, json=data, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()

In [14]:

folder = r"C:\Users\namtv40\Documents\HD\ai-chatbot-excel-files"
HDFS_BASE = "/opt/datasets/crawlers/vcs/excels/data"

if True:
    pathname = folder + r"\Doanh_thu_cong_ty_t11_2025.xlsx"
    df = pd.read_excel(pathname, sheet_name=0)
    RESOURCE_NAME = os.path.basename(pathname)
    RESOURCE_NAME = os.path.splitext(RESOURCE_NAME)[0]
    RESOURCE_NAME = to_sql_column_name(RESOURCE_NAME)
    print(RESOURCE_NAME)
    start_time= time.time()
  
    df = df.where(pd.notnull(df), None)
    df, mapping = normalize_df_columns(df)
    
    records = df.to_dict(orient="records")
    if not records:
        print("No new data")
        sys.exit(0)

    # =========================
    # Pandas → Parquet
    # =========================
    now = datetime.now()
    partition_path = "{}/{}".format(
        HDFS_BASE,
        RESOURCE_NAME
    )

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        RESOURCE_NAME,
        now.year,
        now.month,
        now.day,
        now.hour,
        now.minute,
        now.second
    )
    local_parquet = "./tmp/data/{}/{}".format(RESOURCE_NAME,filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)
    schema = infer_schema_from_json(records)
    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(table, local_parquet, compression="snappy")

    upload_hdfs_https(
        "{}".format(partition_path),
        local_parquet
    )

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(schema=schema,db="dwh", table="raw_"+RESOURCE_NAME, location="{}/{}".format(HDFS_BASE, RESOURCE_NAME))

    filename = "create_table_{}.sql".format(
        RESOURCE_NAME
    )

    local_sql = "./tmp/data/{}/{}".format(RESOURCE_NAME, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)


    print("Uploaded SQL definition")
    
    local_column = "./tmp/data/{}/mapping_columns_{}.json".format(RESOURCE_NAME, RESOURCE_NAME)
    os.makedirs(os.path.dirname(local_column), exist_ok=True)

    with open(local_column, "w", encoding="utf-8") as f:
        f.write(json.dumps(mapping, ensure_ascii=False, indent=4))


    elapsed = time.time() - start_time
    print("Loop {} took {:.3f}s".format(RESOURCE_NAME, elapsed))

doanh_thu_cong_ty_t11_2025
Upload  /opt/datasets/crawlers/vcs/excels/data/doanh_thu_cong_ty_t11_2025 ./tmp/data/doanh_thu_cong_ty_t11_2025/data_doanh_thu_cong_ty_t11_2025_20251225_93603.parquet
Uploaded parquet to /opt/datasets/crawlers/vcs/excels/data/doanh_thu_cong_ty_t11_2025
Uploaded SQL definition
Loop doanh_thu_cong_ty_t11_2025 took 1.306s


Start crawl :  dim_cso_company_fields
dim_cso_company_fields
Last state = 2025-11-27T07:39:17Z
Request to  https://vcs-care.freshdesk.com/api/v2/company_fields {'page': 101, 'per_page': 100, 'include': 'company,contacts_count', 'updated_since': '2025-11-27T07:39:17Z'}
Upload  s3a://vcs-raw/cx-cso-raw/dim_cso_company_fields ./tmp/data/dim_cso_company_fields/data_dim_cso_company_fields_20251224_215621.parquet
Uploaded parquet to s3a://vcs-raw/cx-cso-raw/dim_cso_company_fields
Uploaded SQL definition
Loop dim_cso_company_fields took 4.275s
Start crawl :  dim_cso_agents
dim_cso_agents
Last state = 2025-12-24T14:22:27Z
Request to  https://vcs-care.freshdesk.com/api/v2/agents {'page': 101, 'per_page': 100, 'include': 'user_info', 'updated_since': '2025-12-24T14:22:27Z'}

❌ Lỗi request khi gọi https://vcs-care.freshdesk.com/api/v2/agents: 400 Client Error: Bad Request for url: https://vcs-care.freshdesk.com/api/v2/agents?page=101&per_page=100&include=user_info&updated_since=2025-12-24T14%3A22

SystemExit: 0

c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
folder = r"C:\Users\namtv40\Documents\HD\ai-chatbot-excel-files"